# Multi-Class Network Attack Classification (DoS, DDoS, Port Scan, Brute Force and more)

**IBM SkillsBuild Cyber Security 2026 | BharatCares**

**Goal:** Build a classifier that does not only say *"attack or not"* but also tells **which kind of attack** a network flow belongs to.

**Dataset:** CICIDS2017 (Canadian Institute for Cybersecurity), the day-by-day CSV files.

**Classes:** BENIGN, DoS, DDoS, PortScan, BruteForce (FTP/SSH-Patator), WebAttack, Bot. Very rare attacks (Infiltration, Heartbleed) are dropped automatically if they have too few samples.

**Models compared:** Logistic Regression, Decision Tree, Random Forest, Gradient Boosting.

**Main metric:** *macro* F1-score, which averages the F1 of each class equally. Plain accuracy would hide poor results on the small attack classes.

> **Before running:** put the CICIDS2017 CSV files in a folder called `data` next to this notebook (see `README.md`). The more day-files you add, the more attack types you get. You need at least three different classes.

## 1. Import libraries

In [ ]:
import os
import re
import glob
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("Libraries loaded.")

## 2. Configuration

- `MAX_PER_CLASS`: the full dataset has about 2.8 million flows and some classes are huge (DoS Hulk alone has 230,000). To keep memory and time reasonable, each class is capped at this many randomly chosen flows. Use `None` for no cap.
- `MIN_CLASS_SAMPLES`: classes with fewer samples than this are dropped (a model cannot learn from 11 examples).
- `CLASS_GROUPS`: how the original CICIDS2017 labels are grouped into attack families. The **first** matching rule wins.

In [ ]:
DATA_DIR = "data"
LABEL_COL = "Label"
MAX_PER_CLASS = 50_000
MIN_CLASS_SAMPLES = 100
TEST_SIZE = 0.2
OUTPUT_DIR = "outputs"

CLASS_GROUPS = [               # (pattern found in the original label, family name)
    ("web attack", "WebAttack"),     # brute force / XSS / SQL injection against websites
    ("benign",     "BENIGN"),
    ("ddos",       "DDoS"),          # must come before "dos"
    ("dos",        "DoS"),           # Hulk, GoldenEye, slowloris, Slowhttptest
    ("portscan",   "PortScan"),
    ("port scan",  "PortScan"),
    ("patator",    "BruteForce"),    # FTP-Patator and SSH-Patator
    ("bot",        "Bot"),
    ("infiltration", "Infiltration"),
    ("heartbleed", "Heartbleed"),
]
ID_COLS = ["Flow ID", "Source IP", "Destination IP", "Source Port", "Timestamp"]
os.makedirs(OUTPUT_DIR, exist_ok=True)

def to_group(label):
    l = str(label).strip().lower()
    for pattern, name in CLASS_GROUPS:
        if pattern in l:
            return name
    return str(label).strip()

## 3. Load and clean the data file by file

CICIDS2017 is large, so each file is cleaned and capped **one at a time** instead of loading everything at once. Cleaning per file:
- drop identifier columns (IP addresses, timestamps, flow ID) so the model cannot memorise them,
- drop the duplicated column `Fwd Header Length.1` if present,
- convert infinite values to missing and drop those rows,
- remove duplicate rows,
- group the original labels into attack families.

In [ ]:
files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
if not files:
    raise FileNotFoundError(f"No CSV files found in '{DATA_DIR}'. Download the CICIDS2017 CSV files "
                            "(see README.md) and put them in that folder.")

frames, file_stats, orig_counts = [], [], {}
for f in files:
    # latin-1 reads every byte, which avoids errors on the Thursday web-attack file
    d = pd.read_csv(f, low_memory=False, encoding="latin-1")
    d.columns = d.columns.str.strip()
    if LABEL_COL not in d.columns:
        print(f"Skipping {os.path.basename(f)}: no '{LABEL_COL}' column"); continue
    n_raw = len(d)

    raw_label = d[LABEL_COL].astype(str).str.strip()
    d = d.drop(columns=[LABEL_COL] + [c for c in ID_COLS if c in d.columns]
               + [c for c in d.columns if c.endswith(".1") and c[:-2] in d.columns])
    d = d.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    d["group"] = raw_label.map(to_group)
    d["orig_label"] = raw_label

    bad = d.isna().any(axis=1)
    n_bad = int(bad.sum())
    d = d[~bad]
    n_dup = int(d.duplicated().sum())
    d = d.drop_duplicates()

    for k, v in d.groupby(["orig_label", "group"]).size().items():
        orig_counts[k] = orig_counts.get(k, 0) + int(v)
    d = d.drop(columns=["orig_label"])

    feat = [c for c in d.columns if c != "group"]
    d[feat] = d[feat].astype("float32")
    if MAX_PER_CLASS:   # shuffle, then keep at most MAX_PER_CLASS rows of each class
        d = d.sample(frac=1, random_state=RANDOM_STATE)
        d = d[d.groupby("group").cumcount() < MAX_PER_CLASS]

    frames.append(d)
    file_stats.append({"File": os.path.basename(f), "Raw rows": n_raw, "Inf/NaN removed": n_bad,
                       "Duplicates removed": n_dup, "Rows kept": len(d)})
    print(f"Loaded {os.path.basename(f)}")

df = pd.concat(frames, ignore_index=True, sort=False)
print("\nCombined shape:", df.shape)
pd.DataFrame(file_stats).set_index("File")

In [ ]:
# How the original labels were grouped (counts are after cleaning, before the per-class cap)
mapping = (pd.Series(orig_counts).rename("Flows").rename_axis(["Original label", "Class used"]).reset_index())
mapping["Original label"] = mapping["Original label"].str.encode("ascii", "replace").str.decode("ascii")
mapping.sort_values(["Class used", "Flows"], ascending=[True, False]).reset_index(drop=True)

## 4. Choose the classes and apply the global cap

Classes with too few samples are removed. Then a final cap keeps at most `MAX_PER_CLASS` flows per class across all files.

In [ ]:
if MAX_PER_CLASS:
    df = df.sample(frac=1, random_state=RANDOM_STATE)
    df = df[df.groupby("group").cumcount() < MAX_PER_CLASS]

counts = df["group"].value_counts()
too_few = counts[counts < MIN_CLASS_SAMPLES]
if len(too_few):
    print("Dropped classes with fewer than", MIN_CLASS_SAMPLES, "samples:")
    print(too_few.to_string(), "\n")
df = df[~df["group"].isin(too_few.index)].reset_index(drop=True)

if df["group"].nunique() < 3:
    raise ValueError("Fewer than 3 classes found. Add more CICIDS2017 day files to the data folder "
                     "(Tuesday, Wednesday, Friday files add brute force, DoS, port scan and DDoS).")

others = sorted(c for c in df["group"].unique() if c != "BENIGN")
classes = (["BENIGN"] if "BENIGN" in df["group"].values else []) + others
class_to_id = {c: i for i, c in enumerate(classes)}
df["target"] = df["group"].map(class_to_id)

print("Classes used:", classes)
print("Rows:", len(df))

## 5. Exploratory data analysis

In [ ]:
cnt = df["group"].value_counts().reindex(classes)
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=cnt.values, y=cnt.index, color="#4C72B0", ax=ax)
for i, v in enumerate(cnt.values):
    ax.text(v, i, f" {v:,}", va="center")
ax.set_title("Samples per class")
ax.set_xlabel("Number of flows")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/class_distribution.png", dpi=150)
plt.show()
print((cnt / cnt.sum() * 100).round(2).astype(str) + " %")

In [ ]:
# How do a few features differ between attack types? (log scale because of heavy skew)
wanted = ["Flow Duration", "Flow Packets/s", "Total Fwd Packets", "Packet Length Mean",
          "Flow Bytes/s", "Destination Port"]
feats = [c for c in wanted if c in df.columns][:4]
plot_df = df.sample(n=min(30000, len(df)), random_state=RANDOM_STATE)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, f in zip(axes.ravel(), feats):
    tmp = plot_df[[f, "group"]].copy()
    tmp[f] = np.log1p(tmp[f].clip(lower=0))
    sns.boxplot(data=tmp, x="group", y=f, order=classes, ax=ax, showfliers=False, color="#9ecae1")
    ax.set_title(f"{f} (log1p)")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
for ax in axes.ravel()[len(feats):]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/feature_comparison.png", dpi=150)
plt.show()

## 6. Prepare features and split

A **stratified** split keeps the share of every class the same in the train and test sets, which matters because some classes are small. Constant columns are removed.

In [ ]:
X = df.drop(columns=["group", "target"])
y = df["target"]

const_cols = [c for c in X.columns if X[c].nunique() <= 1]
X = X.drop(columns=const_cols)
print(f"Removed {len(const_cols)} constant columns. Features used: {X.shape[1]}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 7. Train and compare models

Because some classes are much smaller than others, every model uses **class weights** so that mistakes on small classes count more during training.

- **Logistic Regression**: linear baseline (scaled features).
- **Decision Tree**: easy to explain.
- **Random Forest**: many trees voting together.
- **Gradient Boosting (Hist)**: trees built one after another, each correcting the previous ones. Often the strongest on tabular data.

In [ ]:
models = {
    "Logistic Regression": make_pipeline(StandardScaler(),
        LogisticRegression(max_iter=300, class_weight="balanced", random_state=RANDOM_STATE)),
    "Decision Tree": DecisionTreeClassifier(max_depth=20, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced_subsample",
                                            n_jobs=-1, random_state=RANDOM_STATE),
    "Gradient Boosting": HistGradientBoostingClassifier(class_weight="balanced", random_state=RANDOM_STATE),
}

rows, fitted, preds = [], {}, {}
for name, model in models.items():
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    fit_s = time.perf_counter() - t0
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Macro precision": precision_score(y_test, pred, average="macro", zero_division=0),
        "Macro recall": recall_score(y_test, pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_test, pred, average="macro", zero_division=0),
        "Weighted F1": f1_score(y_test, pred, average="weighted", zero_division=0),
        "ROC-AUC (OvR)": roc_auc_score(y_test, proba, multi_class="ovr", average="macro"),
        "Train time (s)": fit_s,
    })
    fitted[name], preds[name] = model, pred
    print(f"Trained {name} in {fit_s:.1f}s")

results = pd.DataFrame(rows).set_index("Model").round(4)
results.to_csv(f"{OUTPUT_DIR}/model_comparison.csv")
results

In [ ]:
metric_cols = ["Accuracy", "Macro precision", "Macro recall", "Macro F1", "Weighted F1"]
lo = max(0, results[metric_cols].min().min() - 0.03)
results[metric_cols].plot(kind="bar", figsize=(10, 4.5), ylim=(lo, 1.005))
plt.title("Model comparison on the test set")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(loc="lower right", ncol=5, fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/model_comparison.png", dpi=150)
plt.show()

## 8. Per-class results of the best model

The best model is chosen by **macro F1**. The table shows precision, recall and F1 for **each** attack type.

In [ ]:
best_name = results["Macro F1"].idxmax()
best_model, best_pred = fitted[best_name], preds[best_name]
print("Best model by macro F1:", best_name, "\n")

rep = pd.DataFrame(classification_report(y_test, best_pred, target_names=classes,
                                         output_dict=True, zero_division=0)).T
rep[["precision", "recall", "f1-score"]] = rep[["precision", "recall", "f1-score"]].round(4)
rep["support"] = rep["support"].astype(int)
rep.to_csv(f"{OUTPUT_DIR}/per_class_report.csv")
rep

In [ ]:
# Per-class F1 for every model: where does each model struggle?
f1_table = pd.DataFrame({name: f1_score(y_test, p, average=None, labels=range(len(classes)), zero_division=0)
                         for name, p in preds.items()}, index=classes)
plt.figure(figsize=(8, 4.8))
sns.heatmap(f1_table, annot=True, fmt=".3f", cmap="YlGnBu", vmin=max(0, f1_table.values.min() - 0.05), vmax=1)
plt.title("F1-score per class and model")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/per_class_f1_heatmap.png", dpi=150)
plt.show()

## 9. Confusion matrix and the most confused attack types

In [ ]:
cm = confusion_matrix(y_test, best_pred, labels=range(len(classes)))
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes,
            cbar=False, ax=axes[0])
axes[0].set_title(f"Confusion matrix (counts): {best_name}")
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=classes, yticklabels=classes,
            cbar=False, ax=axes[1], vmin=0, vmax=1)
axes[1].set_title("Share of each actual class (rows sum to 1)")
for ax in axes:
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# The most common mistakes
pairs = []
for i, a in enumerate(classes):
    for j, p in enumerate(classes):
        if i != j and cm[i, j] > 0:
            pairs.append({"Actual": a, "Predicted as": p, "Flows": int(cm[i, j]),
                          "% of actual class": round(100 * cm_norm[i, j], 2)})
errors = pd.DataFrame(pairs).sort_values("Flows", ascending=False).head(8).reset_index(drop=True)
print(f"Total errors: {int(cm.sum() - np.trace(cm)):,} of {int(cm.sum()):,} test flows")
errors if len(errors) else print("No errors on the test set.")

## 10. Cross-validation (checking the result is stable)

In [ ]:
cv_idx = X_train.sample(n=min(50000, len(X_train)), random_state=RANDOM_STATE).index
y_cv = y_train.loc[cv_idx]
n_splits = int(min(5, y_cv.value_counts().min()))
if n_splits >= 2:
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(best_model, X_train.loc[cv_idx], y_cv, cv=cv, scoring="f1_macro", n_jobs=-1)
    print(f"{n_splits}-fold macro F1 per fold:", scores.round(4))
    print(f"Mean macro F1 = {scores.mean():.4f}  (std = {scores.std():.4f})")
else:
    print("Not enough samples in the smallest class for cross-validation.")

## 11. Which features identify each attack?

First, the overall most important features (Random Forest). Then a heatmap showing, for the top features, whether each attack type has a *higher* (red) or *lower* (blue) typical value than the other classes.

In [ ]:
rf = fitted["Random Forest"]
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
top = imp.head(15)

plt.figure(figsize=(8, 5.5))
sns.barplot(x=top.values, y=top.index, color="#4C72B0")
plt.title("Top 15 features (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/feature_importance.png", dpi=150)
plt.show()
top.round(4)

In [ ]:
top10 = list(imp.head(10).index)
med = X_train[top10].groupby(y_train.map(dict(enumerate(classes)))).median().reindex(classes)
med_log = np.log1p(med.clip(lower=0))
z = (med_log - med_log.mean()) / med_log.std().replace(0, 1)

plt.figure(figsize=(10, 4.8))
sns.heatmap(z, cmap="vlag", center=0, annot=True, fmt=".1f", cbar_kws={"label": "relative level"})
plt.title("Typical value of the top features per class (log scale, standardised)")
plt.ylabel("")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/class_feature_heatmap.png", dpi=150)
plt.show()

## 12. Save the trained model

In [ ]:
joblib.dump({"model": best_model, "features": list(X.columns), "classes": classes, "name": best_name},
            f"{OUTPUT_DIR}/attack_classifier.joblib")
print("Saved:", sorted(os.listdir(OUTPUT_DIR)))

## 13. Try the classifier on a few unseen flows

In [ ]:
sample = X_test.sample(10, random_state=RANDOM_STATE)
proba = best_model.predict_proba(sample)
out = pd.DataFrame({
    "Actual": y_test.loc[sample.index].map(dict(enumerate(classes))),
    "Predicted": [classes[i] for i in proba.argmax(axis=1)],
    "Confidence": proba.max(axis=1).round(3),
}, index=sample.index)
out["Correct"] = out["Actual"] == out["Predicted"]
out

## 14. Conclusion and limitations

- Flow statistics such as packet counts, packet sizes, durations and rates are enough to tell most attack families apart, not only attack from normal traffic.
- Looking at the **per-class** table and the confusion matrix matters more than overall accuracy, because the large classes dominate accuracy while the small classes (for example web attacks and bots) are where models usually fail.

**Limitations (be honest about these in your report):**
1. CICIDS2017 is a **lab-generated** dataset. Researchers have reported problems with traffic generation, flow construction and labelling in it, so very high scores should not be read as real-world performance.
2. Classes were **capped** at `MAX_PER_CLASS` flows, so the class mix in the test set is not the mix of a real network, where benign traffic is by far the majority. Precision on real traffic would be lower.
3. Related attacks were **grouped** (four DoS tools into DoS, three web attacks into WebAttack). Very rare classes (Infiltration, Heartbleed) were dropped, so the model cannot detect them.
4. A random split puts flows from the same attack run in both training and test sets. Testing on a different day or dataset would be a stricter check.
5. Attack tools change over time, so models need retraining on fresh data.

**Future work:** evaluate on a corrected version of the dataset or on CSE-CIC-IDS2018, hold out a whole attack day as a test, add anomaly detection for unseen attack types, and try cost-sensitive thresholds.